[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/twistors/twistors.ipynb)

# Twistors and Linked Light

This example reconstructs null light rays from twistors, builds the twisting Robinson congruence of light rays filling spacetime, and propagates a source-free electromagnetic pulse whose electric and magnetic field lines form linked circles. Every event along a single light ray satisfies one linear condition on that ray's twistor, while one fixed non-null twistor picks out a ray through every event of spacetime whose instantaneous directions trace linked Hopf circles and guide the pulse's energy flow.

Spacetime is modelled inside its six-dimensional conformal algebra, where events are null antivectors and twistors live in the eight-component left ideal cut out by three commuting projectors. Because left multiplication preserves that subspace, conformal rotors act on twistors from one side by `representations(boost, twistor)` while sandwiching events by `boost >> point`, and an intrinsic event reverses a twistor's orientation between `incidence(point, twistor)` and `selection(point, robinson_twistor)`, following [Lasenby, Doran, and Arcaute](https://arxiv.org/abs/math-ph/0603037) for the real twistor ideal and [Swearngin et al.](https://arxiv.org/abs/1302.1431) for the linked hopfion field.

In Penrose's two-spinor notation, a twistor is written as a pair of Weyl spinors $Z^\alpha = (\omega^A, \pi_{A'}) \in \mathbb C^4$ paired with its complex-conjugate dual $\bar Z_\alpha = (\bar\pi_A, \bar\omega^{A'})$ of signature $(2, 2)$, spacetime coordinates are packed into the $2 \times 2$ Hermitian Pauli matrix $x^{AA'} = \frac{1}{\sqrt{2}}(t I + x\sigma_x + y\sigma_y + z\sigma_z)$ for the incidence relation $\omega^A = i x^{AA'}\pi_{A'}$, and reconstructing the linked electromagnetic field translates spinor dyads $\phi_{AB}\epsilon_{A'B'} + \bar\phi_{A'B'}\epsilon_{AB}$ back through Infeld–van der Waerden soldering symbols.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure, show_states
from examples.relativity.twistors import render, scenarios

np.set_printoptions(precision=4, suppress=True)

# Spacetime combined with one positive and one negative conformal direction.
spacetime = Algebra("x+y+z+t-")
ga = spacetime * Algebra("u+v-")
mv = NumpyContext(ga).multivector
exact = ga.exact.multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Plane+
Bivector = ga.gatype.bivector()                                              # Bivector+
Even = ga.gatype.even()                                                      # Even+
Full = ga.gatype.full()                                                      # Full+
Intrinsic = ga.gatype(Full * ga.gatype.antiscalar())                         # Full-
# Conformal points, light rays, spacetime events, spatial displacements, and ray twistors are intrinsic.
Point = ga.gatype.antivector()                                               # Point-
Ray = ga.gatype.antibivector()                                               # Ray-
Event = Plane.derive.within(spacetime).derive.dual                           # Event-
Spatial = Plane.derive.within(Algebra("x+y+z+")).derive.dual                 # Spatial-
Field = Bivector.derive.within(spacetime)                                    # Field+
ExtrinsicTwistor = Full.derive.within(Algebra("y+z+u+"))                     # ExtrinsicTwistor+
Twistor = Intrinsic.derive.within(Algebra("y+z+u+"))                         # Twistor-
# Handedness of the electromagnetic pulse and of the Hopf fibres carrying it.
Helicity = ga.gatype.antipseudoscalar()                                      # Helicity-

# Origin, point at infinity, weight-reading horizon plane, and unit future time displacement.
origin = (mv.v.dual() - mv.u.dual()) / 2                                     # [] Point-
infinity = mv.v.dual() + mv.u.dual()                                         # [] Point-
horizon = -(mv.v + mv.u)                                                     # [] Plane+
time = -mv.t.dual()                                                          # [] Event-

## Twistors in the conformal algebra

Three commuting projectors select an eight-component subspace that left multiplication preserves. Embedding a twistor, multiplying on the left, and reading back out induces three orientation-typed actions: `representations` for extrinsic multivectors such as conformal rotors, `incidence` for an intrinsic event acting on a ray's twistor, and `selection` for an intrinsic event acting on an extrinsic twistor, while the conformal pseudoscalar's representation `volume` squares to minus the identity.

In complex spinor notation, those eight real components form four complex twistor components and `volume` acts as multiplication by $i$.

In [ ]:
# Three commuting projectors halve the state space three times, leaving eight real components.
projector = (1 + exact.x) * (1 + exact.yt) * (1 + exact.zv) / 8              # [] Full+
embedding = Twistor * projector                                              # [] Full- <- Twistor-
extrinsic_embedding = ExtrinsicTwistor * projector                           # [] Full+ <- ExtrinsicTwistor+
readout = (8 * Intrinsic).cast(Twistor)                                      # [] Twistor- <- Full-
extrinsic_readout = (8 * Full).cast(ExtrinsicTwistor)                        # [] ExtrinsicTwistor+ <- Full+
representations = readout(Full * embedding)                                  # [] Twistor- <- (Full+, Twistor-)
incidence = extrinsic_readout(Intrinsic * embedding)                         # [] ExtrinsicTwistor+ <- (Full-, Twistor-)
selection = readout(Intrinsic * extrinsic_embedding)                         # [] Twistor- <- (Full-, ExtrinsicTwistor+)
pseudoscalar = exact.xyztuv                                                  # [] Pseudoscalar+
volume = representations(pseudoscalar)                                       # [] Twistor- <- Twistor-

## Shared twistor along a light ray

Every event along one light ray annihilates the same twistor under `incidence`, together with that twistor turned by `volume`. Multiplying two null-separated conformal points maps into both event kernels at once, so singular value decomposition extracts a shared twistor for the ray between them.

In twistor notation, the two-dimensional real kernel spanned by `twistor` and `volume(twistor)` corresponds to the nonzero complex multiples of one projective twistor.

In [ ]:
emission_coords = scenarios.emission_coords
separation = scenarios.separation

In [ ]:
def conformal_point(event: Event) -> Point:
    """Embedding `event` (`[...] Event-`) on the conformal null cone produces a weight-one point (`[...] Point-`)."""
    return origin + event + event.squared() * infinity / 2                   # [...] Point-


def ray_twistor(first: Point, second: Point) -> Twistor:
    """Singular value decomposition of the two-point product's representation extracts the shared light ray's twistor (`[...] Twistor-`)."""
    family = representations(first * second)                                 # [...] Twistor- <- Twistor-
    states, _, _ = family.svd()                                              # [..., 8] Twistor-
    return states[..., 0]                                                    # [...] Twistor-


emission = mv(Event.derive.dual, emission_coords).dual()                     # [] Event-
light_direction = time + (mv.x.dual() + mv.z.dual()) / np.sqrt(2)            # [] Event-
events = stack([emission, emission + light_direction * separation])          # [2] Event-
points = conformal_point(events)                                             # [2] Point-
twistor = ray_twistor(points[0], points[1])                                  # [] Twistor-

## Reconstructing and boosting a light ray

A conformal-invariant pairing between an intrinsic twistor and an extrinsic twistor turns any null twistor back into its conformal light ray `ray_readout(twistor, twistor)`, the join of its events. Intersecting that ray with constant-time hyperplanes recovers the events along it, whether the boost acts on the twistor from one side or sandwiches the conformal points.

In twistor notation, `pairing` corresponds to the $\mathrm{SU}(2, 2)$-invariant Hermitian form $Z^\alpha \bar W_\alpha$, which vanishes on null twistors.

In [ ]:
rapidities = scenarios.rapidities
n_boosts = len(rapidities)
n_ray_samples = scenarios.n_ray_samples
ray_margin = scenarios.ray_margin

In [ ]:
antiscalar = exact.scalar([1]).dual()                                        # [] Antiscalar-
pairing = 8 * exact.yz.scalar_product(embedding.reverse() * antiscalar * extrinsic_embedding)  # [] Scalar+ <- (Twistor-, ExtrinsicTwistor+)
# Reversing Ray accounts for the reverse-product metric used by contract.
ray_readout = (Ray * pairing(Twistor, incidence(Ray.reverse(), Twistor))).contract(1, 3)  # [] Ray- <- (Twistor-, Twistor-)


def ray_events(ray: Ray, times: Scalar) -> Event:
    """Intersecting `ray` (`[...] Ray-`) with constant-time hyperplanes recovers the spacetime events at `times` (`[...] Event-`)."""
    crossing = (mv.t - horizon * times) ^ ray                                # [...] Point-
    return crossing.cast(Event) / (horizon & crossing)                       # [...] Event-


boost = ((mv.x.dual() & time).dual() * (rapidities / 2)).exp()               # [n_boosts] Even+
twistors = representations(boost, twistor)                                   # [n_boosts] Twistor-
light_rays = ray_readout(twistors, twistors)                                 # [n_boosts] Ray-
boosted_points = boost[:, None] >> points                                    # [n_boosts, 2] Point-
boosted_events = boosted_points.cast(Event) / (horizon & boosted_points)     # [n_boosts, 2] Event-
fractions = np.linspace(-ray_margin, 1 + ray_margin, n_ray_samples)          # [n_ray_samples]
event_times = mv.t & boosted_events                                          # [n_boosts, 2] Scalar+
ray_times = event_times[:, :1] + (event_times[:, 1:] - event_times[:, :1]) * fractions  # [n_boosts, n_ray_samples] Scalar+
rays = ray_events(light_rays[:, None], ray_times)                            # [n_boosts, n_ray_samples] Event-

save_figure(render.incidence_figure(boosted_events, rays), "twistor_incidence");

Orange balls mark the two null-separated events in the rest frame on the left and after a boost along `x` on the right, while blue rods trace the light rays reconstructed directly from the transformed twistors.

## Robinson congruence

Acting on a fixed non-null extrinsic twistor at each spacetime event via `selection(conformal_point(event), robinson_twistor)` selects one null ray through every event. This twisting family of straight light rays is called the Robinson congruence, and at a single instant its spatial directions are tangent to linked Hopf circles on nested tori.

In twistor notation, the Robinson congruence consists of the null twistors $Z^\alpha$ satisfying $Z^\alpha \bar A_\alpha = 0$ for a fixed non-null twistor $A^\alpha$.

In [ ]:
robinson_twistor = 1 + exact.y - exact.zu + exact.yzu                        # [] ExtrinsicTwistor+
polar_angles = scenarios.polar_angles
n_tori = len(polar_angles)
n_circles = scenarios.n_circles
n_fibre_samples = scenarios.n_fibre_samples

In [ ]:
conformal_time = mv.tuv                                                      # [] Trivector+
symmetry_patch = conformal_time.inverse() | mv.y.dual()                      # [] Antibivector-
helicity = mv(Helicity, [1])                                                 # [] Helicity-


def ray_direction(twistor: Twistor) -> Event:
    """Meeting `twistor`'s reconstructed light ray with the horizon plane gives its future null direction at unit time (`[...] Event-`)."""
    ray = ray_readout(twistor, twistor)                                      # [...] Ray-
    tangent = (horizon ^ ray).cast(Event)                                    # [...] Event-
    return tangent / (mv.t & tangent)                                        # [...] Event-


def robinson_direction(event: Event) -> Event:
    """Acting with `conformal_point(event)` on `robinson_twistor` selects the Robinson congruence's future null direction (`[...] Event-`)."""
    selected = selection(conformal_point(event), robinson_twistor)           # [...] Twistor-
    return ray_direction(selected)                                           # [...] Event-


save_figure(render.congruence_figure(scenarios.congruence_lines()), "twistor_robinson");

Coloured tubes trace instantaneous flow curves tangent to the Robinson light-ray directions on three nested tori at time zero, with every pair of closed curves linked once.

## Vacuum electromagnetic hopfion

The Robinson congruence carries a null, source-free electromagnetic pulse whose electric and magnetic field lines form linked Hopf circles. Sandwiching the unit time displacement with the Faraday bivector, `field >> time`, gives the pulse's energy-momentum flow along the observer's time, which aligns at unit time with `robinson_direction` at every event while turning the instantaneous flow curves by a quarter turn in the `xy` plane produces the initial electric field lines.

In Gibbs–Heaviside vector notation, this null field satisfies $\mathbf E \cdot \mathbf B = 0$ and $|\mathbf E| = |\mathbf B|$, with Poynting flux $\mathbf E \times \mathbf B$ directed along the Robinson rays.

In [ ]:
probe = mv(Event.derive.dual, scenarios.probe_coords).dual()                 # [] Event-

In [ ]:
spacetime_pseudoscalar = exact.xyzt                                          # [] Quadvector+


def hopfion_field(event: Event) -> Field:
    """Bateman's spinor construction evaluates the source-free null electromagnetic field (`[...] Field+`) carried by the Robinson congruence."""
    elapsed = mv.t & event                                                   # [...] Scalar+
    # Spatial offset relative to a light flash running along y, contracted into a spatial patch.
    position = conformal_time.inverse() | (event - elapsed * (time + mv.y.dual()))  # [...] Antibivector-
    # Multiplying the spatial patch by helicity gives the spatial bivector turning polarization and propagation together.
    wave_spinor = 1 + position * helicity                                    # [...] Even+
    polarization = (mv.t + mv.y) ^ mv.x                                      # [] Field+
    turned_polarization = wave_spinor >> polarization                        # [...] Field+
    # Spacetime's pseudoscalar rotates electric into magnetic planes while the inverse supplies amplitude decay and phase.
    envelope = (1 + event.squared() + 2 * spacetime_pseudoscalar * elapsed).inverse()  # [...] Even+
    return turned_polarization * envelope.squared() * envelope               # [...] Field+


field = hopfion_field(probe)                                                 # [] Field+
flux = field >> time                                                         # [] Event-
energy_velocity = flux / (mv.t & flux)                                       # [] Event-
probe_direction = robinson_direction(probe)                                  # [] Event-

electric_turn = (mv.xy * (np.pi / 4)).exp()                                  # [] Even+
flow_curves = scenarios.congruence_lines()                                   # [n_tori, n_circles, n_fibre_samples] Spatial-
electric_curves = electric_turn >> flow_curves                               # [n_tori, n_circles, n_fibre_samples] Spatial-

In [ ]:
print(f"energy velocity:\n{energy_velocity.coefficient_table()}")
print(f"Robinson direction:\n{probe_direction.coefficient_table()}")

In [ ]:
save_figure(render.fields_figure(electric_curves), "twistor_hopfion");

Turned by a quarter turn in the `xy` plane relative to the congruence, the linked closed tubes trace the hopfion's electric field lines at time zero.

## Linked field lines in motion

Because the pulse is null, its electric field lines are frozen into the Robinson energy flow: each point on an initial field line travels at the speed of light along its straight ray while the linked loops unfurl and re-form without ever cutting across one another.

In [ ]:
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms
time_span = scenarios.time_span

In [ ]:
frame_times = np.linspace(-time_span, time_span, n_frames)                   # [n_frames]


def propagated_lines(initial_lines: Spatial, frame_times: np.ndarray) -> Iterator[Event]:
    """Yields the linked field lines (`[...] Event-`) transported along their straight Robinson light rays at `frame_times`."""
    directions = robinson_direction(initial_lines)                           # [...] Event-
    for elapsed in frame_times:
        yield initial_lines + directions * elapsed                           # [...] Event-


streamed_lines = show_states(
    propagated_lines(electric_curves, frame_times),
    render.fields_animation,
    name="twistor_hopfion",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
    colors=32,
);

Carried along straight Robinson light rays from `t = -1.25` to `t = 1.25`, the linked electric field loops contract toward the focal region at time zero and expand outward while preserving every pairwise link.

In [ ]:
# checks
# Both events send the ray's twistor to zero, and intersecting the boosted rays with constant-time slices recovers them.
assert (incidence(points, twistor).scalar_norm_squared() < 1e-18).all()
assert ((ray_events(light_rays[:, None], mv.t & boosted_events) - boosted_events).scalar_norm_squared() < 1e-18).all()
# The hopfion field is null, its energy velocity matches the Robinson direction, and all frames streamed.
assert (field.squared().scalar_norm_squared() < 1e-18).all()
assert ((energy_velocity - probe_direction).scalar_norm_squared() < 1e-18).all()
assert len(streamed_lines) == n_frames